In [1]:
import pandas as pd

### Filtering: 2007년 이후

In [2]:
# 파일 로드
file_path = "../dataset/sub-data-notes.csv"
df = pd.read_csv(file_path, dtype=str, encoding="cp949")  # 전체 str로 읽어서 비고 행 보존

# 비고 행 (index=1) 분리
header_row = df.iloc[[0]]  # 비고 행 (실제론 index 0, 칼럼명 행은 pd가 자동으로 헤더로 읽음)
note_row = df.iloc[[0]]    # 다시 확인: read_csv 시 row0=칼럼, row1=비고

# 올바른 분리
note_row = df.iloc[[0]]    # index=1에 해당하는 첫 번째 데이터 행 (비고)
data_rows = df.iloc[1:]    # 나머지 데이터 행들

# YEAR 칼럼에서 2007 초과인 행만 필터
data_filtered = data_rows[pd.to_numeric(data_rows["YEAR"], errors="coerce") > 2007]

# 비고 행 + 필터된 데이터 합치기
result = pd.concat([note_row, data_filtered], ignore_index=True)

# 저장
output_path = "../dataset/sub-data-notes-filtered.csv"
result.to_csv(output_path, index=False, encoding="cp949")

print(f"비고 행 보존 + 데이터 {len(data_filtered)}행 저장됨")
print(result.head())

비고 행 보존 + 데이터 93725행 저장됨
   CASEID          WGT         YEAR   Y1  YY1     X1   XX1     CPI_DEFL  AGE  \
0     NaN          NaN  >2007로만 필터링  NaN  NaN    NaN   NaN          NaN  NaN   
1  145156   7095.38337         2010   11    1               0.904708204   75   
2  145157  7139.730803         2010   12    1               0.904708204   75   
3  145158  7294.390247         2010   13    1               0.904708204   75   
4  145159  6989.430712         2010   14    1               0.904708204   75   

  AGECL  ... HSAVFIN HSAVNFIN EMERGPSTP HPSTPPAY HPSTPLN HPSTPOTH EMERGCUT  \
0   NaN  ...     NaN      NaN       NaN      NaN     NaN      NaN      NaN   
1     6  ...       0        0                  0       0        0        0   
2     6  ...       0        0                  0       0        0        0   
3     6  ...       0        0                  0       0        0        0   
4     6  ...       0        0                  0       0        0        0   

  HCUTFOOD HCUTENT HCUTOT

### Filtering: 유효 칼럼 추출

In [5]:
# 파일 로드
file_path = "../dataset/sub-data-notes-filtered.csv"
df = pd.read_csv(file_path, dtype=str, encoding="cp949")  # 전체 str로 읽어서 비고 행 보존

# 필요한 칼럼명 리스트 (사용자 정의)
desired_columns = ["CASEID", "YEAR", "CPI_DEFL", "NOCHK", "BNKRUPLAST5", "FORECLLAST5", "HHSEX", "OCCAT1", "OCCAT2", "INDCAT", "LF", "HOUSECL", "EDCL", "EDUC", "AGE", "AGECL", "LIFECL", "FAMSTRUCT", "KIDS", "MARRIED", "EXPENSHILO", "WSAVED","SAVRES1","SAVRES2","SAVRES3","SAVRES4","SAVRES5","SAVRES6","SAVRES7","SAVRES8","SAVRES9","ASSET","FIN","NFIN","LIQ","CHECKING","SAVING","MMA","CDS","STOCKS","EQUITY","DEQ","BOND","GOVTBND","OBND","MORTBND","RETQLIQ","RETEQ","IRAKH","YESFINRISK","NOFINRISK"]

# 선택된 칼럼만 추출 (비고 행과 데이터 행 모두에 적용)
# 존재하지 않는 칼럼이 있을 경우를 대비하여 교집합을 사용
available_columns = [col for col in desired_columns if col in df.columns]

if not available_columns:
    print("경고: 원하는 칼럼이 데이터프레임에 존재하지 않습니다. 모든 칼럼이 유지됩니다.")
    selected_df = df
else:
    selected_df = df[available_columns]

# 올바른 분리 (선택된 칼럼만 포함)
note_row = selected_df.iloc[[0]] # index=0에 해당하는 첫 번째 데이터 행 (비고)
data_rows = selected_df.iloc[1:] # 나머지 데이터 행들

data_filtered = data_rows.copy() # 순차적 필터링을 위해 data_rows의 복사본으로 시작

# "NOCHK" value가 1인 row drop
if "NOCHK" in available_columns:
    data_filtered = data_filtered[pd.to_numeric(data_filtered["NOCHK"], errors="coerce") == 0]
else:
    print("경고: 'NOCHK' 칼럼이 선택되지 않아 필터링을 건너뜁니다.")

# "BNKRUPLAST5" value가 1인 row drop
if "BNKRUPLAST5" in available_columns:
    data_filtered = data_filtered[pd.to_numeric(data_filtered["BNKRUPLAST5"], errors="coerce") == 0]
else:
    print("경고: 'BNKRUPLAST5' 칼럼이 선택되지 않아 필터링을 건너뜁니다.")

# "FORECLLAST5" value가 1인 row drop
if "FORECLLAST5" in available_columns:
    data_filtered = data_filtered[pd.to_numeric(data_filtered["FORECLLAST5"], errors="coerce") == 0]
else:
    print("경고: 'FORECLLAST5' 칼럼이 선택되지 않아 필터링을 건너뜁니다.")

# 비고 행 + 필터된 데이터 합치기
result = pd.concat([note_row, data_filtered], ignore_index=True)

# 저장
output_path = "../dataset/sub-data-notes-filtered-selected-columns.csv"
result.to_csv(output_path, index=False, encoding="cp949")

print(f"완료! 비고 행 1개 + 데이터 {len(data_filtered)}행 저장됨")
print("선택된 칼럼:", available_columns)
print(result.head())

완료! 비고 행 1개 + 데이터 83133행 저장됨
선택된 칼럼: ['CASEID', 'YEAR', 'CPI_DEFL', 'NOCHK', 'BNKRUPLAST5', 'FORECLLAST5', 'HHSEX', 'OCCAT1', 'OCCAT2', 'INDCAT', 'LF', 'HOUSECL', 'EDCL', 'EDUC', 'AGE', 'AGECL', 'LIFECL', 'FAMSTRUCT', 'KIDS', 'MARRIED', 'EXPENSHILO', 'WSAVED', 'SAVRES1', 'SAVRES2', 'SAVRES3', 'SAVRES4', 'SAVRES5', 'SAVRES6', 'SAVRES7', 'SAVRES8', 'SAVRES9', 'ASSET', 'FIN', 'NFIN', 'LIQ', 'CHECKING', 'SAVING', 'MMA', 'CDS', 'STOCKS', 'EQUITY', 'DEQ', 'BOND', 'GOVTBND', 'OBND', 'MORTBND', 'RETQLIQ', 'RETEQ', 'IRAKH', 'YESFINRISK', 'NOFINRISK']
   CASEID         YEAR     CPI_DEFL         NOCHK  \
0     NaN  >2007로만 필터링          NaN  1인 rows drop   
1  145156         2010  0.904708204             0   
2  145157         2010  0.904708204             0   
3  145158         2010  0.904708204             0   
4  145159         2010  0.904708204             0   

                                         BNKRUPLAST5 FORECLLAST5  \
0  \=1(파산선고o)의 경우, 투자 운용 자금 없을 것 그리고 우리 타겟층(클라이언트...    \=1 drop 

### Filtering: Anchor User Outlier 제외

In [ ]:
# Anchor User 필터링: FIN이 0이거나 1,000달러 미만인 유저 제외
file_path = "../dataset/sub-data-notes-filtered-selected-columns.csv"
df = pd.read_csv(file_path, dtype=str, encoding="cp949")

# 비고 행 보존
note_row = df.iloc[[0]]
data_rows = df.iloc[1:].copy()

# FIN 숫자형 변환
data_rows["FIN_num"] = pd.to_numeric(data_rows["FIN"], errors="coerce")

# Anchor User 조건 적용: FIN >= 1000
anchor_filtered = data_rows[data_rows["FIN_num"] >= 1000].copy()

# 임시 숫자형 컬럼 제거
anchor_filtered = anchor_filtered.drop(columns=["FIN_num"])

# 비고 행 + 필터된 데이터 재결합
result = pd.concat([note_row, anchor_filtered], ignore_index=True)

# 저장
output_path = "../dataset/sub-data-notes-filtered-selected-columns-anchor-users.csv"
result.to_csv(output_path, index=False, encoding="cp949")